# 温度駆動力を与える。固相液相の**多結晶**シミュレーションー＞GPU計算
# ー＞ https://doi.org/10.1016/j.mtla.2023.101702 の再現
# 固液界面のみ異方性

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float32
from scipy.spatial.transform import Rotation

# =========================
# 0) Settings / Parameters
# =========================
nx, ny = 256, 256
number_of_grain = 4         # 0: liquid, 1..N-1: solid grains
dx, dy = 1e-4, 1e-4
dt = 0.001
nsteps = 200000             # まず短め推奨（動作確認用）
pi = np.pi

delta = 10.0 * dx
T_melt = 1687.15
G = 1.0e+02
V_pulling = 5.0e-05
latent = 4.15e+05

# ---- Energetic anisotropy (Appendix A / Table values you had) ----
a0 = 54.7 * pi / 180.0
delta_a = 0.36
mu_a = 0.6156
p_round = 0.05

# ---- Interface energies (scale you want) ----
# Solid-Liquid base energy (use (100) baseline)
gamma_100 = 0.44
# Grain boundary energy (isotropic)
gamma_GB = 0.60

# ---- Mobilities (choose reasonable constants) ----
# Grain boundary mobility (isotropic)
M_GB = 3.0e-5
# Solid-liquid mobility (isotropic here; you can later add b(theta))
M_SL = 3.0e-5

# Output
out_dir = "result/periodic/ani_quat"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# =====================================
# 1) Grain orientations (quaternions)
# =====================================
np.random.seed(42)
N = number_of_grain
grain_quaternions = np.zeros((N, 4), dtype=np.float64)
grain_quaternions[0] = np.array([0, 0, 0, 1.0])  # liquid dummy (x,y,z,w)

for gid in range(1, N):
    q = np.random.normal(0, 1, 4)
    q = q / np.linalg.norm(q)
    grain_quaternions[gid] = q

# ---- Precompute rotated {111} normals (8) for each grain ----
n111_base = np.array([
    [ 1,  1,  1], [ 1,  1, -1], [ 1, -1,  1], [-1,  1,  1],
    [ 1, -1, -1], [-1,  1, -1], [-1, -1,  1], [-1, -1, -1],
], dtype=np.float32) / np.sqrt(3.0)

grain_n111 = np.zeros((N, 8, 3), dtype=np.float32)
for gid in range(N):
    Rm = Rotation.from_quat(grain_quaternions[gid]).as_matrix().astype(np.float32)
    grain_n111[gid] = (Rm @ n111_base.T).T  # (8,3)

In [ ]:

# ==========================================================
# 2) Constant GB interactions (wij,aij,mij)  (i,j>0 constant)
#    and baseline SL constants (for reference only)
# ==========================================================
wij = np.zeros((N, N), dtype=np.float32)
aij = np.zeros((N, N), dtype=np.float32)
mij = np.zeros((N, N), dtype=np.float32)

In [ ]:
# --- helper conversions ---
def eps_from_gamma(gamma):
    # epsilon = sqrt(8*delta*gamma/pi^2)
    return math.sqrt(8.0 * delta * gamma / (pi * pi))

def w_from_gamma(gamma):
    # w = 4*gamma/delta
    return 4.0 * gamma / delta

def mij_from_M(M):
    # mij(phi) = (pi^2/(8*delta)) * M   (same form you used)
    return (pi * pi / (8.0 * delta)) * M

# ---- GB constants (isotropic) ----
eps_GB = eps_from_gamma(gamma_GB)
w_GB = w_from_gamma(gamma_GB)
m_GB_phi = mij_from_M(M_GB)

for i in range(1, N):
    for j in range(1, N):
        if i == j:
            continue
        wij[i, j] = w_GB
        aij[i, j] = eps_GB
        mij[i, j] = m_GB_phi

# ---- SL baseline (100) ----
eps0_sl = eps_from_gamma(gamma_100)
w0_sl = w_from_gamma(gamma_100)
m_sl_phi = mij_from_M(M_SL)

for i in range(1, N):
    # store baseline (will be overwritten locally in kernel for anisotropy)
    wij[0, i] = wij[i, 0] = w0_sl
    aij[0, i] = aij[i, 0] = eps0_sl
    mij[0, i] = mij[i, 0] = m_sl_phi

In [ ]:
NG = 4

@cuda.jit(device=True, inline=True)
def calc_a_from_cos(cosT, a0, delta_a, mu_a, p_round):
    # Appendix A (A2)-(A4)
    c2 = cosT*cosT
    C = math.sqrt(c2 + p_round*p_round)
    S = math.sqrt(max(1.0 - c2, 0.0) + p_round*p_round)
    return mu_a * (1.0 + delta_a * (C + math.tan(a0)*S))
@cuda.jit(device=True, inline=True)
def calc_a_theta_from_costh(cosT, a0, delta_a, mu_a, p_round):
    c2 = cosT * cosT
    C = math.sqrt(c2 + p_round * p_round)
    S = math.sqrt(max(1.0 - c2, 0.0) + p_round * p_round)
    return mu_a * (1.0 + delta_a * (C + math.tan(a0) * S))
@cuda.jit(device=True, inline=True)
def grad_phi(phi, gid, l, m, l_p, l_m, m_p, m_m, dx):
    gx = (phi[gid, l_p, m] - phi[gid, l_m, m]) * (0.5/dx)
    gy = (phi[gid, l, m_p] - phi[gid, l, m_m]) * (0.5/dx)  # dx=dy
    return gx, gy

@cuda.jit(device=True, inline=True)
def cos_theta_from_grad(gx, gy, n111, gid):
    # cosθ = max_t | n · n111_rotated |
    g = math.sqrt(gx*gx + gy*gy) + 1e-20
    nx = gx/g
    ny = gy/g
    cmax = 0.0
    for t in range(8):
        n111x = n111[gid, t, 0]
        n111y = n111[gid, t, 1]
        c = abs(nx*n111x + ny*n111y)  # 2D projection
        if c > cmax:
            cmax = c
    if cmax < 0.0: cmax = 0.0
    if cmax > 1.0: cmax = 1.0
    return cmax


In [ ]:
@cuda.jit(device=True, inline=True)
def aniso_operator_A1(phi, gid, l, m, l_p, l_m, m_p, m_m,nx,ny,
                      dx, eps0_sl,  # epsilon0 of SL (100)
                      a0, delta_a, mu_a, p_round,
                      n111):
    inv_dx2 = 1.0/(dx*dx)

    # ---- center grad ----
    gx, gy = grad_phi(phi, gid, l, m, l_p, l_m, m_p, m_m, dx)
    g2 = gx*gx + gy*gy
    g  = math.sqrt(g2) + 1e-20

    # choose nearest {111} (store the best normal components too)
    nx = gx/g
    ny = gy/g
    best = 0.0
    best_nx = 1.0
    best_ny = 0.0
    for t in range(8):
        n111x = n111[gid, t, 0]
        n111y = n111[gid, t, 1]
        c = abs(nx*n111x + ny*n111y)
        if c > best:
            best = c
            # keep the *signed* projection direction consistent with abs()
            # We'll use u = (n·g)/g then abs later; sign handled by using abs(u)
            best_nx = n111x
            best_ny = n111y
    cosT = best
    # ---- a, epsilon, epsilon^2 at center ----
    a = calc_a_theta_from_costh(cosT, a0, delta_a, mu_a, p_round)
    eps = eps0_sl * a
    eps2 = eps * eps

    # =========================================================
    # Term-1: div(eps^2 grad phi)  (flux form)
    # =========================================================
    # neighbor eps2 (evaluate eps2 at neighbor centers)
    # NOTE: this is the "correct" way (eps varies in space).
    # It is heavier but required for Appendix A.
    # --- eps2 at x+ ---
    gx_xp, gy_xp = grad_phi(phi, gid, l_p, m, (l_p+1 if l_p<nx-1 else 0), l, m_p, m_m, dx)
    cos_xp = cos_theta_from_grad(gx_xp, gy_xp, n111, gid)
    a_xp = calc_a_theta_from_costh(cos_xp, a0, delta_a, mu_a, p_round)
    eps2_xp = (eps0_sl*a_xp)*(eps0_sl*a_xp)

    # --- eps2 at x- ---
    gx_xm, gy_xm = grad_phi(phi, gid, l_m, m, l, (l_m-1 if l_m>0 else nx-1), m_p, m_m, dx)
    cos_xm = cos_theta_from_grad(gx_xm, gy_xm, n111, gid)
    a_xm = calc_a_theta_from_costh(cos_xm, a0, delta_a, mu_a, p_round)
    eps2_xm = (eps0_sl*a_xm)*(eps0_sl*a_xm)

    # --- eps2 at y+ ---
    gx_yp, gy_yp = grad_phi(phi, gid, l, m_p, l_p, l_m, (m_p+1 if m_p<ny-1 else m_p), m, dx)
    cos_yp = cos_theta_from_grad(gx_yp, gy_yp, n111, gid)
    a_yp = calc_a_theta_from_costh(cos_yp, a0, delta_a, mu_a, p_round)
    eps2_yp = (eps0_sl*a_yp)*(eps0_sl*a_yp)

    # --- eps2 at y- ---
    gx_ym, gy_ym = grad_phi(phi, gid, l, m_m, l_p, l_m, m, (m_m-1 if m_m>0 else 0), dx)
    cos_ym = cos_theta_from_grad(gx_ym, gy_ym, n111, gid)
    a_ym = calc_a_theta_from_costh(cos_ym, a0, delta_a, mu_a, p_round)
    eps2_ym = (eps0_sl*a_ym)*(eps0_sl*a_ym)

    # fluxes for term-1
    # Fx = eps^2 * dphi/dx at half nodes
    Fx_p = 0.5*(eps2 + eps2_xp) * (phi[gid, l_p, m] - phi[gid, l, m]) / dx
    Fx_m = 0.5*(eps2 + eps2_xm) * (phi[gid, l, m] - phi[gid, l_m, m]) / dx
    Fy_p = 0.5*(eps2 + eps2_yp) * (phi[gid, l, m_p] - phi[gid, l, m]) / dx
    Fy_m = 0.5*(eps2 + eps2_ym) * (phi[gid, l, m] - phi[gid, l, m_m]) / dx

    term1 = (Fx_p - Fx_m)/dx + (Fy_p - Fy_m)/dx

    # =========================================================
    # Term-2 (torque): div( eps * eps_{phi_p} * phi_p / |grad|^2 )
    # =========================================================
    # Need eps_{phi_x}, eps_{phi_y} at center.
    # First compute dcos/dgx,dcos/dgy using best normal components.
    u = best_nx*gx + best_ny*gy
    au = abs(u) + 1e-20
    # cos = abs(u)/g
    # d(abs(u))/dgx = sign(u)*best_nx
    sgn = 1.0 if u >= 0.0 else -1.0
    du_dgx = sgn * best_nx
    du_dgy = sgn * best_ny

    dcos_dgx = (du_dgx*g - au*(gx/g)) / (g*g)
    dcos_dgy = (du_dgy*g - au*(gy/g)) / (g*g)

    # da/dcos
    C = math.sqrt(cosT*cosT + p_round*p_round)
    S = math.sqrt(max(1.0 - cosT*cosT, 0.0) + p_round*p_round)
    da_dcos = mu_a * delta_a * ( (cosT/C) + math.tan(a0)*(-cosT/S) )

    da_dgx = da_dcos * dcos_dgx
    da_dgy = da_dcos * dcos_dgy

    deps_dgx = eps0_sl * da_dgx
    deps_dgy = eps0_sl * da_dgy

    # torque flux components at center
    inv_g2 = 1.0/(g2 + 1e-20)
    Qx = eps * deps_dgx * gx * inv_g2
    Qy = eps * deps_dgy * gy * inv_g2

    # Compute Q at neighbors (central difference for divergence)
    # (Same procedure but cheaper approx: use neighbor grads & a, but reuse best normal search)
    # --- Qx at x+ ---
    gx2, gy2 = gx_xp, gy_xp
    g2p = gx2*gx2 + gy2*gy2
    gp  = math.sqrt(g2p) + 1e-20
    nxp, nyp = gx2/gp, gy2/gp
    bestp=0.0; bnx=1.0; bny=0.0
    for t in range(8):
        n111x = n111[gid,t,0]; n111y = n111[gid,t,1]
        c = abs(nxp*n111x + nyp*n111y)
        if c>bestp:
            bestp=c; bnx=n111x; bny=n111y
    ap = calc_a_theta_from_costh(bestp, a0, delta_a, mu_a, p_round)
    epsp = eps0_sl * ap
    up = bnx*gx2 + bny*gy2
    sgnp = 1.0 if up>=0.0 else -1.0
    dux = sgnp*bnx
    # dcos/dgx at x+ (same formula)
    aup = abs(up)+1e-20
    dcos_dgx_p = (dux*gp - aup*(gx2/gp))/(gp*gp)
    Cp = math.sqrt(bestp*bestp + p_round*p_round)
    Sp = math.sqrt(max(1.0-bestp*bestp,0.0)+p_round*p_round)
    da_dcos_p = mu_a*delta_a*((bestp/Cp)+math.tan(a0)*(-bestp/Sp))
    deps_dgx_p = eps0_sl*(da_dcos_p*dcos_dgx_p)
    Qx_p = epsp*deps_dgx_p*gx2/(g2p+1e-20)

    # --- Qx at x- ---
    gx2, gy2 = gx_xm, gy_xm
    g2m = gx2*gx2 + gy2*gy2
    gm  = math.sqrt(g2m) + 1e-20
    nxm, nym = gx2/gm, gy2/gm
    bestm=0.0; bnx=1.0; bny=0.0
    for t in range(8):
        n111x = n111[gid,t,0]; n111y = n111[gid,t,1]
        c = abs(nxm*n111x + nym*n111y)
        if c>bestm:
            bestm=c; bnx=n111x; bny=n111y
    am = calc_a_theta_from_costh(bestm, a0, delta_a, mu_a, p_round)
    epsm = eps0_sl * am
    um = bnx*gx2 + bny*gy2
    sgnm = 1.0 if um>=0.0 else -1.0
    dux = sgnm*bnx
    aum = abs(um)+1e-20
    dcos_dgx_m = (dux*gm - aum*(gx2/gm))/(gm*gm)
    Cm = math.sqrt(bestm*bestm + p_round*p_round)
    Sm = math.sqrt(max(1.0-bestm*bestm,0.0)+p_round*p_round)
    da_dcos_m = mu_a*delta_a*((bestm/Cm)+math.tan(a0)*(-bestm/Sm))
    deps_dgx_m = eps0_sl*(da_dcos_m*dcos_dgx_m)
    Qx_m = epsm*deps_dgx_m*gx2/(g2m+1e-20)

    # --- Qy at y+ ---
    gx2, gy2 = gx_yp, gy_yp
    g2p = gx2*gx2 + gy2*gy2
    gp  = math.sqrt(g2p) + 1e-20
    nxp, nyp = gx2/gp, gy2/gp
    bestp=0.0; bnx=1.0; bny=0.0
    for t in range(8):
        n111x = n111[gid,t,0]; n111y = n111[gid,t,1]
        c = abs(nxp*n111x + nyp*n111y)
        if c>bestp:
            bestp=c; bnx=n111x; bny=n111y
    ap = calc_a_theta_from_costh(bestp, a0, delta_a, mu_a, p_round)
    epsp = eps0_sl * ap
    up = bnx*gx2 + bny*gy2
    sgnp = 1.0 if up>=0.0 else -1.0
    duy = sgnp*bny
    aup = abs(up)+1e-20
    dcos_dgy_p = (duy*gp - aup*(gy2/gp))/(gp*gp)
    Cp = math.sqrt(bestp*bestp + p_round*p_round)
    Sp = math.sqrt(max(1.0-bestp*bestp,0.0)+p_round*p_round)
    da_dcos_p = mu_a*delta_a*((bestp/Cp)+math.tan(a0)*(-bestp/Sp))
    deps_dgy_p = eps0_sl*(da_dcos_p*dcos_dgy_p)
    Qy_p = epsp*deps_dgy_p*gy2/(g2p+1e-20)

    # --- Qy at y- ---
    gx2, gy2 = gx_ym, gy_ym
    g2m = gx2*gx2 + gy2*gy2
    gm  = math.sqrt(g2m) + 1e-20
    nxm, nym = gx2/gm, gy2/gm
    bestm=0.0; bnx=1.0; bny=0.0
    for t in range(8):
        n111x = n111[gid,t,0]; n111y = n111[gid,t,1]
        c = abs(nxm*n111x + nym*n111y)
        if c>bestm:
            bestm=c; bnx=n111x; bny=n111y
    am = calc_a_theta_from_costh(bestm, a0, delta_a, mu_a, p_round)
    epsm = eps0_sl * am
    um = bnx*gx2 + bny*gy2
    sgnm = 1.0 if um>=0.0 else -1.0
    duy = sgnm*bny
    aum = abs(um)+1e-20
    dcos_dgy_m = (duy*gm - aum*(gy2/gm))/(gm*gm)
    Cm = math.sqrt(bestm*bestm + p_round*p_round)
    Sm = math.sqrt(max(1.0-bestm*bestm,0.0)+p_round*p_round)
    da_dcos_m = mu_a*delta_a*((bestm/Cm)+math.tan(a0)*(-bestm/Sm))
    deps_dgy_m = eps0_sl*(da_dcos_m*dcos_dgy_m)
    Qy_m = epsm*deps_dgy_m*gy2/(g2m+1e-20)

    term2 = (Qx_p - Qx_m)/(2.0*dx) + (Qy_p - Qy_m)/(2.0*dx)

    return term1 + term2


In [ ]:
@cuda.jit
def kernel_update_temp(temp, cooling_rate, nx, ny):
    l, m = cuda.grid(2)
    if l < nx and m < ny:
        temp[l, m] -= cooling_rate

In [ ]:
NG = 4

@cuda.jit
def kernel_update_phasefield(phi, phi_new, temp,
                             wij, aij, mij,
                             n111,
                             nx, ny, number_of_grain, dx, dt,
                             T_melt, latent,
                             eps0_sl, w0_sl,
                             a0, delta_a, mu_a, p_round):

    l, m = cuda.grid(2)
    if l >= nx or m >= ny:
        return

    l_p = l + 1 if l < nx - 1 else 0
    l_m = l - 1 if l > 0 else nx - 1
    m_p = m + 1 if m < ny - 1 else m
    m_m = m - 1 if m > 0 else 0

    inv_dx2 = 1.0 / (dx * dx)

    lap_k = cuda.local.array(NG, dtype=float32)
    phi_c = cuda.local.array(NG, dtype=float32)

    for k in range(number_of_grain):
        pc = phi[k, l, m]
        phi_c[k] = pc
        lap_k[k] = (phi[k, l_p, m] + phi[k, l_m, m] +
                    phi[k, l, m_p] + phi[k, l, m_m] -
                    4.0 * pc) * inv_dx2

    # ---- 単相判定を i-loop の外で一回だけやる ----
    threshold = 1e-4  # まずはこれくらい推奨
    num_phase = 0
    for k in range(number_of_grain):
        if (phi[k, l, m]   > threshold or
            phi[k, l_p, m] > threshold or
            phi[k, l_m, m] > threshold or
            phi[k, l, m_p] > threshold or
            phi[k, l, m_m] > threshold):
            num_phase += 1

    if num_phase <= 1:
        for i in range(number_of_grain):
            phi_new[i, l, m] = phi[i, l, m]
        return

    # ---- 支配固相を選ぶ（界面法線用）----
    i_s = 1
    maxv = phi_c[1]
    for g in range(2, number_of_grain):
        v = phi_c[g]
        if v > maxv:
            maxv = v
            i_s = g

    phix = (phi[i_s, l_p, m] - phi[i_s, l_m, m]) * (0.5 / dx)
    phiy = (phi[i_s, l, m_p] - phi[i_s, l, m_m]) * (0.5 / dx)
    gn = math.sqrt(phix * phix + phiy * phiy) + 1e-20
    nx_ = phix / gn
    ny_ = phiy / gn

    cmax = 0.0
    for t in range(8):
        n111x = n111[i_s, t, 0]
        n111y = n111[i_s, t, 1]
        c = abs(nx_ * n111x + ny_ * n111y)
        if c > cmax:
            cmax = c
    if cmax < 0.0: cmax = 0.0
    if cmax > 1.0: cmax = 1.0

    a_loc = calc_a_theta_from_costh(cmax, a0, delta_a, mu_a, p_round)
    a2_loc = a_loc * a_loc
    w_sl    = w0_sl * a2_loc

    curr_temp = temp[l, m]
    for i in range(number_of_grain):
        dpi = 0.0
        for j in range(number_of_grain):
            if i == j:
                continue

            # ---- driving force: ΔS(T-Tm) ----
            DeltaS = latent / T_melt
            curr_temp = temp[l, m]
            driving_force = 0.0
            if i != 0 and j == 0:
                driving_force = -DeltaS * (curr_temp - T_melt)   # solid grows when T<Tm
            elif i == 0 and j != 0:
                driving_force =  DeltaS * (curr_temp - T_melt)

            ppp = 0.0
            for k in range(number_of_grain):
                wik = wij[i, k]
                wjk = wij[j, k]

                # --- SL: wを異方性にする（重要） ---
                if k == 0:
                    if i != 0:
                        wik = w_sl
                    if j != 0:
                        wjk = w_sl

                term1 = (wik - wjk) * phi_c[k]
                term2 = 0.0  # デフォルトは入れない（後で必要なときだけ）

                # ---- gradient項：基本は等方 ----
                # ただし固液ペアでは「固相側のk」だけを扱う
                if (j == 0 and i != 0):
                    # solid i vs liquid
                    if k == i:
                        lap_use = aniso_operator_A1(phi, i, l, m, l_p, l_m, m_p, m_m,nx,ny,
                                                    dx, eps0_sl, a0, delta_a, mu_a, p_round, n111)
                        # A1は「既にepsを含む」ので係数差は 1 とみなす
                        term2 = 0.5 * (1.0 - 0.0) * lap_use
                    else:
                        term2 = 0.0  # 他相は混ぜない
                elif (i == 0 and j != 0):
                    # liquid vs solid j
                    if k == j:
                        lap_use = aniso_operator_A1(phi, j, l, m, l_p, l_m, m_p, m_m,nx,ny,
                                                    dx, eps0_sl, a0, delta_a, mu_a, p_round, n111)
                        term2 = 0.5 * (0.0 - 1.0) * lap_use
                    else:
                        term2 = 0.0
                else:
                    # solid-solid: 等方のまま
                    eps2ik = aij[i, k] * aij[i, k]
                    eps2jk = aij[j, k] * aij[j, k]
                    term2 = 0.5 * (eps2ik - eps2jk) * lap_k[k]

                ppp += term1 + term2

            phii_phij = phi[i, l, m] * phi[j, l, m]
            term_force = (8.0 / 3.1415926535) * math.sqrt(max(phii_phij, 0.0)) * driving_force
            dpi -= (2.0 / float(num_phase)) * mij[i, j] * (ppp - term_force)

        phi_new[i, l, m] = phi[i, l, m] + dpi * dt

In [ ]:
# =====================================================
# 4) Initialization
# =====================================================
phi_cpu = np.zeros((number_of_grain, nx, ny), dtype=np.float32)
seed_height = 32
factor = 2.0 / delta

n_solid = number_of_grain - 1
grain_width = nx // n_solid

for l in range(nx):
    grain_id = int(l // grain_width) + 1
    if grain_id > n_solid:
        grain_id = n_solid
    for m in range(ny):
        y = m * dy
        dist = y - (seed_height * dy)
        phi_solid = 0.5 * (1.0 - np.tanh(factor * dist))
        phi_cpu[grain_id, l, m] = phi_solid
        phi_cpu[0, l, m] = 1.0 - phi_solid

temp_cpu = np.zeros((nx, ny), dtype=np.float64)
for m in range(ny):
    temp_cpu[:, m] = T_melt - 1.0 + G * (m - seed_height) * dy

# quick plot step0
phase_map = np.argmax(phi_cpu, axis=0)
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(phase_map.T, cmap="tab20", origin="lower", interpolation="nearest")
plt.colorbar(ticks=range(number_of_grain), label="Phase ID")
plt.title("Initial Grain Map")

plt.subplot(1, 3, 2)
plt.imshow(temp_cpu.T, cmap="hot", origin="lower")
plt.colorbar(label="T [K]")
plt.title("Initial Temperature")

plt.subplot(1, 3, 3)
plt.imshow(phi_cpu[1].T, cmap="bwr", origin="lower", vmin=0, vmax=1)
plt.colorbar()
plt.title("phi(grain 1) step0")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "step_0.png"), dpi=150)
plt.close()

In [ ]:
# dtypeを統一（推奨）
phi_cpu  = phi_cpu.astype(np.float32)
temp_cpu = temp_cpu.astype(np.float32)

d_phi     = cuda.to_device(phi_cpu)
d_phi_new = cuda.to_device(phi_cpu)
d_temp    = cuda.to_device(temp_cpu)
d_wij     = cuda.to_device(wij.astype(np.float32))
d_aij     = cuda.to_device(aij.astype(np.float32))
d_mij     = cuda.to_device(mij.astype(np.float32))
d_n111    = cuda.to_device(grain_n111.astype(np.float32))

threadsperblock = (16, 16)
blockspergrid = (math.ceil(nx / threadsperblock[0]),
                 math.ceil(ny / threadsperblock[1]))

cooling_rate = np.float32(G * V_pulling * dt * 0.0)  # 温度固定なら0でOK
T_melt_f  = np.float32(T_melt)
latent_f  = np.float32(latent)
# もし kernel 側を ΔS入力に直すなら
# DeltaS_f = np.float32(latent / T_melt)

In [ ]:
# =====================================================
# 6) Main loop
# =====================================================
save_every = 10000

for nstep in range(1, nsteps + 1):
    kernel_update_temp[blockspergrid, threadsperblock](d_temp, cooling_rate, nx, ny)

    kernel_update_phasefield[blockspergrid, threadsperblock](
        d_phi, d_phi_new, d_temp,
        d_wij, d_aij, d_mij,
        d_n111,
        nx, ny, number_of_grain, np.float32(dx), np.float32(dt),
        T_melt_f, latent_f,
        np.float32(eps0_sl), np.float32(w0_sl),
        np.float32(a0), np.float32(delta_a), np.float32(mu_a), np.float32(p_round)
    )

    d_phi, d_phi_new = d_phi_new, d_phi

    if nstep % save_every == 0:
        current_phi  = d_phi.copy_to_host()
        current_temp = d_temp.copy_to_host()
        print(f"Step {nstep} | Tmin={current_temp.min():.3f} Tmax={current_temp.max():.3f}")

        plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1)
        plt.imshow(phase_map.T, cmap="tab20", origin="lower", interpolation="nearest")
        plt.colorbar(ticks=range(number_of_grain), label="Phase ID")
        plt.title(f"Phases step {nstep}")

        plt.subplot(1, 3, 2)
        plt.imshow(current_temp.T, cmap="hot", origin="lower")
        plt.colorbar(label="T [K]")
        plt.title(f"Temperature step {nstep}")

        plt.subplot(1, 3, 3)
        plt.imshow(current_phi[1].T, cmap="bwr", origin="lower", vmin=0, vmax=1)
        plt.colorbar()
        plt.title(f"phi(grain 1) step {nstep}")

        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"step_{nstep}.png"), dpi=150)
        plt.close()
